In [10]:
import boto3                                                     #addr fom. ANKIT
import json
import re
from collections import defaultdict
from postal.parser import parse_address

# =====================
# CONFIG
# =====================

ENDPOINT_NAME = "indicbert-address-ner-v3"
runtime = boto3.client("sagemaker-runtime")

# =====================
# Endpoint Call
# =====================

def call_endpoint(text):
    payload = json.dumps({"inputs": text})
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=payload,
    )
    return json.loads(response["Body"].read())

# =====================
# Merge subwords
# =====================

def merge_subword_predictions(preds):
    results = []
    current = None
    buf = ""
    scores = []
    start = None
    end = None

    def flush():
        nonlocal current, buf, scores, start, end
        if current:
            results.append({
                "entity": current,
                "word": buf.strip(),
                "start": start,
                "end": end,
                "score": sum(scores) / len(scores),
            })
        current = None
        buf = ""
        scores = []
        start = None
        end = None

    for p in preds:
        tag = p["entity"]
        token = p["word"]
        clean = token.replace("▁", "")

        if tag.startswith("B-"):
            flush()
            current = tag[2:]
            buf = clean
            scores = [p["score"]]
            start = p["start"]
            end = p["end"]

        elif tag.startswith("I-") and current == tag[2:]:
            if token.startswith("▁"):
                buf += " "
            buf += clean
            end = p["end"]
            scores.append(p["score"])
        else:
            flush()

    flush()
    return results

# =====================
# Smart repair
# =====================

def smart_repair_entities(spans):
    fixed = []
    for s in spans:
        word = s["word"].strip()
        ent = s["entity"]
        new_ent = ent

        if re.fullmatch(r"\d{6}", word):
            new_ent = "PINCODE"
        elif re.search(r"[A-Za-z]", word) and not re.search(r"\d", word):
            if ent == "PINCODE":
                new_ent = "CITY"
        if re.search(r"(lp|flat|plot|house|unit|room|no\.?)", word, re.I):
            new_ent = "PREMISES_NO"

        if word.lower() in ["road", "lane", "street", "sarani", "avenue"]: 
            new_ent = "ROAD"
        
        fixed.append({**s, "entity": new_ent})
    return fixed

# =====================
# Libpostal
# =====================

def parse_with_libpostal(text):
    parsed = parse_address(text)
    structured = defaultdict(str)
    for val, label in parsed:
        structured[label] += (" " + val)
    return dict(structured)

# =====================
# Final formatter
# =====================

def format_address_sentence(structured):
    def normalize(text):
        return text.strip().title()  # Capitalize properly

    house = normalize(structured.get("house_number", ""))
    road = normalize(structured.get("road", ""))
    city = normalize(structured.get("city", ""))
    postcode = structured.get("postcode", "").strip()

    parts = []
    
    if house:
        parts.append(house)
    if road:
        parts.append(road)
    if city:
        parts.append(city)
    if postcode:
        parts.append(postcode)

    # Clean up punctuation
    sentence = ", ".join([p for p in parts if p])
    sentence = re.sub(r"\s+", " ", sentence).strip()

    return sentence
    
# ===============================================
def validate_address(structured):
    house = structured.get("house_number", "").strip()
    road = structured.get("road", "").strip()
    city = structured.get("city", "").strip()
    postcode = structured.get("postcode", "").strip()

    # Require at least one strong field
    if not (house or road or city or postcode):
        return False

    # PIN code check (India: 6 digits)
    if postcode and not re.fullmatch(r"\d{6}", postcode):
        return False

    return True

# =====================
# MAIN
# =====================

def extract_and_format_address(text):
    preds = call_endpoint(text)
    merged = merge_subword_predictions(preds)
    repaired = smart_repair_entities(merged)

    # feed to libpostal as ordered string
    ordered = ", ".join([s["word"] for s in sorted(repaired, key=lambda x: x["start"])])
    structured = parse_with_libpostal(ordered)
    
    if not validate_address(structured):
        final_sentence = "invalid I/P"
    else:
        final_sentence = format_address_sentence(structured)
    
    if len(final_sentence) > 120:
        final_sentence = "invalid I/P"


    return final_sentence

# =====================
# TEST (Real-time input)
# =====================

if __name__ == "__main__":
    user_text = input("Please put addres in the following format { House No., Street, City & Pin Code }: ")

    structured, final_sentence = extract_and_format_address(user_text)


    print("FINAL ADDRESS:")
    print(final_sentence)
    #print(structured)

Please put addres in the following format { House No., Street, City & Pin Code }:  ddd, hjjjk, huhh, 697987


FINAL ADDRESS:
invalid I/P
{'house': ' ddd h huhh'}
